# Linear Regression from Scratch

Interview prep: implement linear regression in pure NumPy

## What we'll cover:
1. The math (MSE loss, gradient)
2. Closed-form solution (Normal Equation)
3. Gradient Descent solution
4. By-hand calculation
5. Interview questions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: The Math

### Model
```
ŷ = Xw + b
```

Or with bias absorbed into weights:
```
ŷ = X_b @ w  (where X_b has column of 1s prepended)
```

### Loss Function (Mean Squared Error)
```
L = (1/n) Σ(yᵢ - ŷᵢ)²
  = (1/n) ||y - Xw||²
```

### Gradient
```
∂L/∂w = (2/n) Xᵀ(Xw - y)
      = (2/n) Xᵀ(ŷ - y)
      = -(2/n) Xᵀ(y - ŷ)
```

## Part 2: Closed-Form Solution (Normal Equation)

Set gradient to zero and solve:
```
∂L/∂w = 0
Xᵀ(Xw - y) = 0
XᵀXw = Xᵀy
w = (XᵀX)⁻¹Xᵀy
```

This is the **Normal Equation**.

In [ ]:
def fit_closed_form(X, y):
    """
    Closed-form solution: w = (XᵀX)⁻¹Xᵀy
    
    Args:
        X: (n_samples, n_features)
        y: (n_samples,)
    
    Returns:
        weights: (n_features,)
        bias: scalar
    """
    n_samples = X.shape[0]
    
    # Add bias column (column of 1s)
    X_b = np.hstack([np.ones((n_samples, 1)), X])
    
    # Normal equation: w = (XᵀX)⁻¹Xᵀy
    # Using np.linalg.solve is more stable than computing inverse
    XtX = X_b.T @ X_b
    Xty = X_b.T @ y
    
    w = np.linalg.solve(XtX, Xty)
    
    bias = w[0]
    weights = w[1:]
    
    return weights, bias

In [ ]:
# Test closed-form solution
# True relationship: y = 2*x1 + 3*x2 + 5

X = np.random.randn(100, 2)
y = 2 * X[:, 0] + 3 * X[:, 1] + 5

weights, bias = fit_closed_form(X, y)

print(f"True weights: [2, 3], bias: 5")
print(f"Learned weights: {weights.round(6)}, bias: {bias:.6f}")

## Part 3: Gradient Descent Solution

Iteratively update weights:
```
w = w - lr × gradient
w = w - lr × (2/n) × Xᵀ(ŷ - y)
```

In [ ]:
def fit_gradient_descent(X, y, lr=0.01, n_iters=1000, verbose=False):
    """
    Gradient descent solution.
    
    Args:
        X: (n_samples, n_features)
        y: (n_samples,)
        lr: learning rate
        n_iters: number of iterations
    
    Returns:
        weights, bias, loss_history
    """
    n_samples, n_features = X.shape
    
    # Initialize
    weights = np.zeros(n_features)
    bias = 0
    loss_history = []
    
    for i in range(n_iters):
        # Predict
        y_pred = X @ weights + bias
        
        # Error
        error = y_pred - y
        
        # Loss
        loss = np.mean(error ** 2)
        loss_history.append(loss)
        
        # Gradients
        dw = (2 / n_samples) * (X.T @ error)
        db = (2 / n_samples) * np.sum(error)
        
        # Update
        weights -= lr * dw
        bias -= lr * db
        
        if verbose and i % 100 == 0:
            print(f"Iter {i}: Loss = {loss:.6f}")
    
    return weights, bias, loss_history

In [ ]:
# Test gradient descent
X = np.random.randn(100, 2)
y = 2 * X[:, 0] + 3 * X[:, 1] + 5

weights_gd, bias_gd, losses = fit_gradient_descent(X, y, lr=0.1, n_iters=500, verbose=True)

print(f"\nTrue weights: [2, 3], bias: 5")
print(f"GD weights: {weights_gd.round(6)}, bias: {bias_gd:.6f}")

In [ ]:
# Plot loss curve
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Loss Curve')

plt.subplot(1, 2, 2)
plt.plot(losses[:50])
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Loss Curve (first 50 iters)')

plt.tight_layout()
plt.show()

## Part 4: By-Hand Calculation

Simple example:
```
X = [[1],    y = [3,
     [2],         5,
     [3]]         7]

Find w and b: y = w*x + b
```

In [ ]:
# By hand calculation
X_simple = np.array([[1], [2], [3]])
y_simple = np.array([3, 5, 7])

print("Step 1: Add bias column")
X_b = np.hstack([np.ones((3, 1)), X_simple])
print(f"X_b = \n{X_b}")

print("\nStep 2: Compute XᵀX")
XtX = X_b.T @ X_b
print(f"XᵀX = \n{XtX}")

print("\nStep 3: Compute Xᵀy")
Xty = X_b.T @ y_simple
print(f"Xᵀy = {Xty}")

print("\nStep 4: Solve XᵀX @ w = Xᵀy")
w = np.linalg.solve(XtX, Xty)
print(f"w = {w}")

print(f"\nResult: y = {w[1]:.1f}*x + {w[0]:.1f}")
print(f"Check: {w[1]}*1 + {w[0]} = {w[1]*1 + w[0]}")
print(f"Check: {w[1]}*2 + {w[0]} = {w[1]*2 + w[0]}")
print(f"Check: {w[1]}*3 + {w[0]} = {w[1]*3 + w[0]}")

## Part 5: Full Class Implementation

In [ ]:
class LinearRegression:
    """
    Linear Regression from scratch.
    
    Methods:
    - fit_closed_form: w = (XᵀX)⁻¹Xᵀy
    - fit_gradient_descent: iterative optimization
    """
    
    def __init__(self):
        self.weights = None
        self.bias = None
    
    def fit_closed_form(self, X, y):
        """Closed-form solution."""
        n_samples = X.shape[0]
        X_b = np.hstack([np.ones((n_samples, 1)), X])
        
        w = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)
        
        self.bias = w[0]
        self.weights = w[1:]
        return self
    
    def fit_gradient_descent(self, X, y, lr=0.01, n_iters=1000):
        """Gradient descent solution."""
        n_samples, n_features = X.shape
        
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(n_iters):
            y_pred = X @ self.weights + self.bias
            error = y_pred - y
            
            self.weights -= lr * (2 / n_samples) * (X.T @ error)
            self.bias -= lr * (2 / n_samples) * np.sum(error)
        
        return self
    
    def predict(self, X):
        """Predict."""
        return X @ self.weights + self.bias
    
    def mse(self, X, y):
        """Mean Squared Error."""
        return np.mean((y - self.predict(X)) ** 2)
    
    def r2_score(self, X, y):
        """R² score (coefficient of determination)."""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

In [ ]:
# Test the class
X = np.random.randn(100, 3)
y = 1.5 * X[:, 0] - 2 * X[:, 1] + 0.5 * X[:, 2] + 3 + np.random.randn(100) * 0.1

# Closed form
model_cf = LinearRegression()
model_cf.fit_closed_form(X, y)

# Gradient descent
model_gd = LinearRegression()
model_gd.fit_gradient_descent(X, y, lr=0.1, n_iters=1000)

print("True weights: [1.5, -2, 0.5], bias: 3")
print(f"\nClosed form:")
print(f"  Weights: {model_cf.weights.round(4)}")
print(f"  Bias: {model_cf.bias:.4f}")
print(f"  R²: {model_cf.r2_score(X, y):.4f}")

print(f"\nGradient Descent:")
print(f"  Weights: {model_gd.weights.round(4)}")
print(f"  Bias: {model_gd.bias:.4f}")
print(f"  R²: {model_gd.r2_score(X, y):.4f}")

## Part 6: Ridge Regression (L2 Regularization)

Add L2 penalty to prevent overfitting:

```
L = MSE + λ||w||²

Closed form: w = (XᵀX + λI)⁻¹Xᵀy
```

In [ ]:
def fit_ridge(X, y, alpha=1.0):
    """
    Ridge regression: w = (XᵀX + αI)⁻¹Xᵀy
    
    Args:
        X: (n_samples, n_features)
        y: (n_samples,)
        alpha: regularization strength
    
    Returns:
        weights, bias
    """
    n_samples = X.shape[0]
    X_b = np.hstack([np.ones((n_samples, 1)), X])
    n_features = X_b.shape[1]
    
    # Regularization matrix (don't regularize bias)
    I = np.eye(n_features)
    I[0, 0] = 0  # Don't regularize bias term
    
    # Ridge solution
    w = np.linalg.solve(X_b.T @ X_b + alpha * I, X_b.T @ y)
    
    return w[1:], w[0]

# Test ridge
X = np.random.randn(100, 2)
y = 2 * X[:, 0] + 3 * X[:, 1] + 5 + np.random.randn(100) * 0.5

for alpha in [0, 0.1, 1.0, 10.0]:
    w, b = fit_ridge(X, y, alpha=alpha)
    print(f"α={alpha:4.1f}: weights={w.round(3)}, bias={b:.3f}")

## Part 7: Interview Questions

### Q1: When does closed-form fail?

In [ ]:
# When XᵀX is singular (not invertible)
# Example: more features than samples

X_bad = np.random.randn(5, 10)  # 5 samples, 10 features
y_bad = np.random.randn(5)

try:
    weights, bias = fit_closed_form(X_bad, y_bad)
    print(f"Weights: {weights}")
except np.linalg.LinAlgError as e:
    print(f"Error: {e}")

# Solution: use Ridge (regularization makes XᵀX + αI invertible)
weights, bias = fit_ridge(X_bad, y_bad, alpha=0.1)
print(f"Ridge weights: {weights.round(3)}")

### Q2: Gradient Descent vs Closed Form

| | Closed Form | Gradient Descent |
|---|---|---|
| Complexity | O(n³) matrix inversion | O(n × features × iters) |
| Large data | Slow | Fast |
| Features > Samples | Fails | Works |
| Non-linear loss | No | Yes |
| Exact solution | Yes | Approximate |

### Q3: What's R²?

```
R² = 1 - (SS_res / SS_tot)

SS_res = Σ(y - ŷ)²   (unexplained variance)
SS_tot = Σ(y - ȳ)²   (total variance)

R² = 1: perfect fit
R² = 0: same as predicting mean
R² < 0: worse than mean (bad model)
```

In [ ]:
# R² demonstration
X = np.random.randn(100, 1)
y = 2 * X[:, 0] + 1 + np.random.randn(100) * 0.5

model = LinearRegression()
model.fit_closed_form(X, y)

y_pred = model.predict(X)
y_mean = np.mean(y)

ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y_mean) ** 2)

r2 = 1 - (ss_res / ss_tot)

print(f"SS_res (unexplained): {ss_res:.2f}")
print(f"SS_tot (total):       {ss_tot:.2f}")
print(f"R² = 1 - {ss_res:.2f}/{ss_tot:.2f} = {r2:.4f}")

### Q4: Assumptions of Linear Regression

1. **Linearity**: y is linear combination of features
2. **Independence**: observations are independent
3. **Homoscedasticity**: constant variance of errors
4. **Normality**: errors are normally distributed
5. **No multicollinearity**: features not highly correlated

### Q5: Derive the gradient

In [ ]:
# Verify gradient numerically
X = np.random.randn(10, 2)
y = np.random.randn(10)
w = np.random.randn(2)
b = 0.5

# Analytical gradient
y_pred = X @ w + b
error = y_pred - y
grad_analytical = (2 / len(y)) * (X.T @ error)

# Numerical gradient
eps = 1e-5
grad_numerical = np.zeros_like(w)
for i in range(len(w)):
    w_plus = w.copy()
    w_plus[i] += eps
    loss_plus = np.mean((X @ w_plus + b - y) ** 2)
    
    w_minus = w.copy()
    w_minus[i] -= eps
    loss_minus = np.mean((X @ w_minus + b - y) ** 2)
    
    grad_numerical[i] = (loss_plus - loss_minus) / (2 * eps)

print(f"Analytical gradient: {grad_analytical.round(6)}")
print(f"Numerical gradient:  {grad_numerical.round(6)}")
print(f"Match: {np.allclose(grad_analytical, grad_numerical)}")

## Summary: What to Remember

### Closed Form
```python
X_b = np.hstack([np.ones((n, 1)), X])  # add bias column
w = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)
```

### Gradient Descent
```python
y_pred = X @ w + b
error = y_pred - y
dw = (2/n) * X.T @ error
db = (2/n) * np.sum(error)
w -= lr * dw
b -= lr * db
```

### Ridge
```python
w = np.linalg.solve(X_b.T @ X_b + alpha * I, X_b.T @ y)
```

### Metrics
```python
MSE = np.mean((y - y_pred) ** 2)
R2 = 1 - SS_res / SS_tot
```